# RT Notebook 18  
## Does Every Domain Projection Necessarily Generate Residue?

### Theory Series II — Falsification Notebook

**Primary hypothesis**

\[
H_1:\quad
\forall \Pi_{D_s\rightarrow D_t},
\quad D_s\neq D_t
\Rightarrow
R(\Pi)>0
\]

> Every genuine projection between distinct domains necessarily generates residue.

**Null / falsification condition**

\[
H_0:\quad
\exists \Pi_{D_s\rightarrow D_t}
\text{ such that }
D_s\neq D_t
\land
R(\Pi)=0
\]

A single admissible zero-residue cross-domain projection falsifies the universal hypothesis **for the tested model class**.

This notebook does not assume the hypothesis is true. It searches directly for counterexamples.

## Why residue must be operationalized before testing

“Residue” can mean several different failures of preservation. This notebook therefore measures a **residue vector** rather than forcing one preferred definition:

\[
R(\Pi)=
(R_{\mathrm{collision}},
R_{\mathrm{orientation}},
R_{\mathrm{relation}},
R_{\mathrm{reference}},
R_{\mathrm{inverse}},
R_{\mathrm{type}})
\]

The notebook reports results under four increasingly strong residue criteria:

1. **Information residue**  
   Loss through collisions, orientation conflict, relation loss, or non-invertibility.

2. **Structural residue**  
   Failure to preserve orientation and relational organization.

3. **Reference residue**  
   A target organization requires a reference choice not fixed by the source.

4. **Typed-domain residue**  
   Source and target domains have different declared admissible structure, even if the finite map is bijective.

The universal hypothesis is evaluated separately under each criterion so it cannot be “saved” merely by changing the definition after the search.

# Deliverables

When run, the notebook produces:

- `outputs_notebook18/projection_records.csv`
- `outputs_notebook18/projection_summary.csv`
- `outputs_notebook18/zero_residue_counterexamples.csv`
- `outputs_notebook18/residue_component_rates.csv`
- `outputs_notebook18/figure_residue_rates.png`
- `outputs_notebook18/figure_zero_residue_by_domain_pair.png`
- `outputs_notebook18/findings18.json`
- `outputs_notebook18/run_manifest18.json`
- `outputs_notebook18/RT_Notebook_18_outputs.zip`

The outputs include exhaustive results for the declared finite model class, explicit counterexamples, interpretation, limitations, and a recommendation for Notebook 19.

# Model Class

A domain instance is represented as a finite relational organization:

\[
O=(V,\sigma,E,\rho,\tau)
\]

where:

- \(V\): finite members,
- \(\sigma\): binary orientation labels,
- \(E\): undirected relational edges,
- \(\rho\): optional distinguished reference member,
- \(\tau\): declared domain type.

A projection is a total deterministic function:

\[
\Pi:V_s\rightarrow V_t
\]

The exhaustive campaign searches:

- all binary orientation assignments;
- all undirected graph organizations;
- all admissible reference choices;
- all total maps between tested source and target sizes;
- multiple domain-type pairs.

The finite search cannot prove a universal theorem over all possible domains. It can:

1. falsify the hypothesis by finding a counterexample;
2. support it within a clearly declared bounded model class;
3. identify which residue components are necessary and which are definition-dependent.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from itertools import combinations, product
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
from collections import Counter, defaultdict
import hashlib
import json
import math
import os
import platform
import random
import statistics
import sys
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 180018
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path("outputs_notebook18")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output directory:", OUTPUT_DIR.resolve())

## 1. Domain and organization definitions

In [ ]:
@dataclass(frozen=True)
class DomainType:
    name: str
    dof: int
    orientation_required: bool
    relations_required: bool
    reference_required: bool
    description: str


DOMAIN_TYPES = {
    "core": DomainType(
        name="core",
        dof=0,
        orientation_required=False,
        relations_required=False,
        reference_required=False,
        description="Primitive distinction condition with no imposed internal orientation."
    ),
    "ordinal": DomainType(
        name="ordinal",
        dof=1,
        orientation_required=True,
        relations_required=False,
        reference_required=False,
        description="Ordinal orientation is admissible."
    ),
    "gradient": DomainType(
        name="gradient",
        dof=2,
        orientation_required=True,
        relations_required=True,
        reference_required=True,
        description="Orientation, relation, and reference structure are admissible."
    ),
    "temporal": DomainType(
        name="temporal",
        dof=3,
        orientation_required=True,
        relations_required=True,
        reference_required=True,
        description="Temporal interpretation is locally primitive."
    ),
}


@dataclass(frozen=True)
class Organization:
    domain: str
    size: int
    orientation: Tuple[int, ...]
    edges: Tuple[Tuple[int, int], ...]
    reference: Optional[int]

    def __post_init__(self):
        if self.domain not in DOMAIN_TYPES:
            raise ValueError(f"Unknown domain: {self.domain}")
        if len(self.orientation) != self.size:
            raise ValueError("Orientation length must equal organization size.")
        if self.reference is not None and not (0 <= self.reference < self.size):
            raise ValueError("Reference must be a valid member index.")

    def signature(self) -> Tuple:
        return (
            self.domain,
            self.size,
            self.orientation,
            self.edges,
            self.reference,
        )

    def uid(self) -> str:
        raw = json.dumps(self.signature(), sort_keys=True).encode("utf-8")
        return hashlib.sha256(raw).hexdigest()[:16]


def all_edges(n: int) -> Tuple[Tuple[int, int], ...]:
    return tuple(combinations(range(n), 2))


def powerset_edges(n: int) -> Iterable[Tuple[Tuple[int, int], ...]]:
    candidates = all_edges(n)
    for mask in range(1 << len(candidates)):
        yield tuple(candidates[i] for i in range(len(candidates)) if mask & (1 << i))


def generate_organizations(domain_name: str, n: int) -> Iterable[Organization]:
    dtype = DOMAIN_TYPES[domain_name]

    orientation_space = (
        product((-1, 1), repeat=n)
        if dtype.orientation_required
        else [tuple(0 for _ in range(n))]
    )

    edge_space = list(powerset_edges(n)) if dtype.relations_required else [tuple()]
    reference_space = range(n) if dtype.reference_required else [None]

    for orientation in orientation_space:
        for edges in edge_space:
            for reference in reference_space:
                yield Organization(
                    domain=domain_name,
                    size=n,
                    orientation=tuple(orientation),
                    edges=tuple(sorted(edges)),
                    reference=reference,
                )


for name in DOMAIN_TYPES:
    count = sum(1 for _ in generate_organizations(name, 3))
    print(f"{name:8s} organizations at n=3: {count}")

## 2. Projection and residue definitions

In [ ]:
@dataclass(frozen=True)
class ResidueVector:
    collision: int
    orientation: int
    relation: int
    reference: int
    inverse: int
    type_mismatch: int

    @property
    def information_total(self) -> int:
        return (
            self.collision
            + self.orientation
            + self.relation
            + self.inverse
        )

    @property
    def structural_total(self) -> int:
        return self.orientation + self.relation

    @property
    def reference_total(self) -> int:
        return self.reference

    @property
    def typed_total(self) -> int:
        return (
            self.collision
            + self.orientation
            + self.relation
            + self.reference
            + self.inverse
            + self.type_mismatch
        )


def all_total_maps(n_source: int, n_target: int) -> Iterable[Tuple[int, ...]]:
    return product(range(n_target), repeat=n_source)


def map_is_injective(mapping: Tuple[int, ...]) -> bool:
    return len(set(mapping)) == len(mapping)


def map_is_surjective(mapping: Tuple[int, ...], n_target: int) -> bool:
    return set(mapping) == set(range(n_target))


def projected_source_orientation(
    source: Organization,
    mapping: Tuple[int, ...],
    n_target: int,
) -> Tuple[Optional[int], ...]:
    buckets: List[List[int]] = [[] for _ in range(n_target)]
    for s, t in enumerate(mapping):
        buckets[t].append(source.orientation[s])

    projected = []
    for values in buckets:
        if not values:
            projected.append(None)
        elif all(v == values[0] for v in values):
            projected.append(values[0])
        else:
            projected.append(0)
    return tuple(projected)


def projected_source_edges(
    source: Organization,
    mapping: Tuple[int, ...],
) -> set[Tuple[int, int]]:
    image_edges = set()
    for a, b in source.edges:
        x, y = mapping[a], mapping[b]
        if x != y:
            image_edges.add(tuple(sorted((x, y))))
    return image_edges


def compute_residue(
    source: Organization,
    target: Organization,
    mapping: Tuple[int, ...],
) -> ResidueVector:
    if len(mapping) != source.size:
        raise ValueError("Mapping length must equal source size.")

    collision = source.size - len(set(mapping))

    projected_orientation = projected_source_orientation(
        source, mapping, target.size
    )
    orientation_residue = 0
    for idx, projected_value in enumerate(projected_orientation):
        if projected_value is None:
            if DOMAIN_TYPES[target.domain].orientation_required:
                orientation_residue += 1
        elif projected_value == 0:
            orientation_residue += 1
        elif projected_value != target.orientation[idx]:
            orientation_residue += 1

    image_edges = projected_source_edges(source, mapping)
    target_edges = set(target.edges)
    relation_residue = len(image_edges.symmetric_difference(target_edges))

    if source.reference is None and target.reference is None:
        reference_residue = 0
    elif source.reference is None and target.reference is not None:
        reference_residue = 1
    elif source.reference is not None and target.reference is None:
        reference_residue = 1
    else:
        reference_residue = int(mapping[source.reference] != target.reference)

    inverse_residue = int(
        not (
            source.size == target.size
            and map_is_injective(mapping)
            and map_is_surjective(mapping, target.size)
        )
    )

    source_type = DOMAIN_TYPES[source.domain]
    target_type = DOMAIN_TYPES[target.domain]
    type_mismatch = int(
        (
            source_type.orientation_required,
            source_type.relations_required,
            source_type.reference_required,
            source_type.dof,
        )
        !=
        (
            target_type.orientation_required,
            target_type.relations_required,
            target_type.reference_required,
            target_type.dof,
        )
    )

    return ResidueVector(
        collision=collision,
        orientation=orientation_residue,
        relation=relation_residue,
        reference=reference_residue,
        inverse=inverse_residue,
        type_mismatch=type_mismatch,
    )

## 3. Admissibility rules

The hypothesis becomes meaningless if every arbitrary map is called a projection. This notebook therefore tests two projection classes.

### Weakly admissible projection

A total map between organizations in distinct declared domains.

### Strongly admissible projection

A weakly admissible projection that also:

- covers every target member;
- maps the source reference to the target reference when both exist;
- does not invent target orientation where a source image exists;
- does not invent target relations beyond the projected source relations.

The strong class is deliberately conservative. Results are reported for both classes.

In [ ]:
def weakly_admissible(
    source: Organization,
    target: Organization,
    mapping: Tuple[int, ...],
) -> bool:
    return (
        source.domain != target.domain
        and len(mapping) == source.size
        and all(0 <= x < target.size for x in mapping)
    )


def strongly_admissible(
    source: Organization,
    target: Organization,
    mapping: Tuple[int, ...],
) -> bool:
    if not weakly_admissible(source, target, mapping):
        return False

    if not map_is_surjective(mapping, target.size):
        return False

    if source.reference is not None and target.reference is not None:
        if mapping[source.reference] != target.reference:
            return False

    projected_orientation = projected_source_orientation(
        source, mapping, target.size
    )
    for idx, value in enumerate(projected_orientation):
        if value is not None and value != 0:
            if target.orientation[idx] != value:
                return False

    image_edges = projected_source_edges(source, mapping)
    if not set(target.edges).issubset(image_edges):
        return False

    return True

## 4. Exhaustive campaign configuration

In [ ]:
CONFIG = {
    "source_domains": ["core", "ordinal", "gradient"],
    "target_domains": ["ordinal", "gradient", "temporal"],
    "source_sizes": [1, 2, 3],
    "target_sizes": [1, 2, 3],
    "max_records": None,
    "seed": SEED,
}

DOMAIN_PAIRS = [
    (s, t)
    for s in CONFIG["source_domains"]
    for t in CONFIG["target_domains"]
    if s != t
]

print("Domain pairs:")
for pair in DOMAIN_PAIRS:
    print(" ", pair)

### Computational note

The complete Cartesian product of all organizations can become large. The campaign is exhaustive over all maps and all source organizations in the declared size range. For target organizations, it evaluates all organizations when the pair is small and uses deterministic stratified enumeration when a pair exceeds the safety threshold.

The run manifest records whether each domain-size block was exhaustive or stratified.

In [ ]:
MAX_TARGET_ORGS_PER_BLOCK = 500

def deterministic_target_subset(
    organizations: Sequence[Organization],
    limit: int,
) -> List[Organization]:
    if len(organizations) <= limit:
        return list(organizations)

    ordered = sorted(organizations, key=lambda o: o.uid())
    step = len(ordered) / limit
    indices = sorted({min(len(ordered)-1, int(i * step)) for i in range(limit)})
    return [ordered[i] for i in indices]


def campaign_blocks():
    for source_domain, target_domain in DOMAIN_PAIRS:
        for ns in CONFIG["source_sizes"]:
            for nt in CONFIG["target_sizes"]:
                source_orgs = list(generate_organizations(source_domain, ns))
                all_target_orgs = list(generate_organizations(target_domain, nt))
                target_orgs = deterministic_target_subset(
                    all_target_orgs,
                    MAX_TARGET_ORGS_PER_BLOCK,
                )
                yield {
                    "source_domain": source_domain,
                    "target_domain": target_domain,
                    "source_size": ns,
                    "target_size": nt,
                    "source_orgs": source_orgs,
                    "target_orgs": target_orgs,
                    "target_total": len(all_target_orgs),
                    "target_used": len(target_orgs),
                    "target_mode": (
                        "exhaustive"
                        if len(target_orgs) == len(all_target_orgs)
                        else "deterministic_stratified"
                    ),
                }


preview = []
for block in campaign_blocks():
    preview.append({
        k: v for k, v in block.items()
        if k not in {"source_orgs", "target_orgs"}
    })

pd.DataFrame(preview).head(10)

## 5. Execute the residue search

In [ ]:
records = []
block_manifest = []
start_time = time.time()

for block_index, block in enumerate(campaign_blocks(), start=1):
    source_domain = block["source_domain"]
    target_domain = block["target_domain"]
    ns = block["source_size"]
    nt = block["target_size"]

    local_count = 0
    strong_count = 0

    for source in block["source_orgs"]:
        mappings = list(all_total_maps(ns, nt))
        for target in block["target_orgs"]:
            for mapping in mappings:
                if not weakly_admissible(source, target, mapping):
                    continue

                residue = compute_residue(source, target, mapping)
                strong = strongly_admissible(source, target, mapping)

                records.append({
                    "source_domain": source_domain,
                    "target_domain": target_domain,
                    "source_size": ns,
                    "target_size": nt,
                    "source_uid": source.uid(),
                    "target_uid": target.uid(),
                    "source_orientation": json.dumps(source.orientation),
                    "target_orientation": json.dumps(target.orientation),
                    "source_edges": json.dumps(source.edges),
                    "target_edges": json.dumps(target.edges),
                    "source_reference": source.reference,
                    "target_reference": target.reference,
                    "mapping": json.dumps(mapping),
                    "strongly_admissible": strong,
                    "collision_residue": residue.collision,
                    "orientation_residue": residue.orientation,
                    "relation_residue": residue.relation,
                    "reference_residue": residue.reference,
                    "inverse_residue": residue.inverse,
                    "type_mismatch_residue": residue.type_mismatch,
                    "information_residue": residue.information_total,
                    "structural_residue": residue.structural_total,
                    "typed_residue": residue.typed_total,
                    "zero_information_residue": residue.information_total == 0,
                    "zero_structural_residue": residue.structural_total == 0,
                    "zero_reference_residue": residue.reference_total == 0,
                    "zero_typed_residue": residue.typed_total == 0,
                })
                local_count += 1
                strong_count += int(strong)

    block_manifest.append({
        "block_index": block_index,
        "source_domain": source_domain,
        "target_domain": target_domain,
        "source_size": ns,
        "target_size": nt,
        "source_organizations": len(block["source_orgs"]),
        "target_organizations_total": block["target_total"],
        "target_organizations_used": block["target_used"],
        "target_mode": block["target_mode"],
        "weak_projection_records": local_count,
        "strong_projection_records": strong_count,
    })

elapsed = time.time() - start_time
df = pd.DataFrame(records)

print(f"Projection records: {len(df):,}")
print(f"Elapsed seconds: {elapsed:,.2f}")
print(f"Strongly admissible: {int(df['strongly_admissible'].sum()):,}")

## 6. Validation checks

In [ ]:
VALIDATION = {}

VALIDATION["records_exist"] = len(df) > 0
VALIDATION["all_cross_domain"] = bool(
    (df["source_domain"] != df["target_domain"]).all()
)
VALIDATION["residue_nonnegative"] = bool(
    (
        df[
            [
                "collision_residue",
                "orientation_residue",
                "relation_residue",
                "reference_residue",
                "inverse_residue",
                "type_mismatch_residue",
            ]
        ] >= 0
    ).all().all()
)
VALIDATION["typed_residue_consistent"] = bool(
    (
        df["typed_residue"]
        ==
        df[
            [
                "collision_residue",
                "orientation_residue",
                "relation_residue",
                "reference_residue",
                "inverse_residue",
                "type_mismatch_residue",
            ]
        ].sum(axis=1)
    ).all()
)
VALIDATION["strong_subset_of_weak"] = True
VALIDATION["domain_pairs_present"] = len(
    df[["source_domain", "target_domain"]].drop_duplicates()
) == len(DOMAIN_PAIRS)

for key, value in VALIDATION.items():
    print(f"{key:32s}: {'PASS' if value else 'FAIL'}")

assert all(VALIDATION.values()), "Validation failed."

## 7. Primary falsification test

In [ ]:
CRITERIA = {
    "information": "zero_information_residue",
    "structural": "zero_structural_residue",
    "reference": "zero_reference_residue",
    "typed": "zero_typed_residue",
}

test_rows = []

for admissibility_name, subset in {
    "weak": df,
    "strong": df[df["strongly_admissible"]],
}.items():
    for criterion_name, zero_col in CRITERIA.items():
        total = len(subset)
        zero_count = int(subset[zero_col].sum())
        test_rows.append({
            "admissibility": admissibility_name,
            "criterion": criterion_name,
            "tested_projections": total,
            "zero_residue_counterexamples": zero_count,
            "counterexample_rate": zero_count / total if total else np.nan,
            "universal_hypothesis_survives": zero_count == 0 if total else None,
        })

test_results = pd.DataFrame(test_rows)
test_results

### Decision rule

For each residue criterion and admissibility class:

- **FALSIFIED** if at least one zero-residue cross-domain projection is found.
- **NOT FALSIFIED IN TESTED MODEL CLASS** if none is found.
- **UNTESTED** if no admissible projections exist.

“Not falsified” is not reported as universal proof.

In [ ]:
def verdict(row):
    if row["tested_projections"] == 0:
        return "UNTESTED"
    if row["zero_residue_counterexamples"] > 0:
        return "FALSIFIED_IN_TESTED_MODEL_CLASS"
    return "NOT_FALSIFIED_IN_TESTED_MODEL_CLASS"

test_results["verdict"] = test_results.apply(verdict, axis=1)
test_results

## 8. Counterexample extraction

In [ ]:
counterexample_frames = []

for admissibility_name, subset in {
    "weak": df,
    "strong": df[df["strongly_admissible"]],
}.items():
    for criterion_name, zero_col in CRITERIA.items():
        found = subset[subset[zero_col]].copy()
        if not found.empty:
            found.insert(0, "criterion", criterion_name)
            found.insert(0, "admissibility", admissibility_name)
            counterexample_frames.append(found)

if counterexample_frames:
    counterexamples = pd.concat(counterexample_frames, ignore_index=True)
else:
    counterexamples = pd.DataFrame()

print("Counterexample rows:", len(counterexamples))
counterexamples.head(20)

## 9. Residue component rates

In [ ]:
component_cols = [
    "collision_residue",
    "orientation_residue",
    "relation_residue",
    "reference_residue",
    "inverse_residue",
    "type_mismatch_residue",
]

component_rows = []

for admissibility_name, subset in {
    "weak": df,
    "strong": df[df["strongly_admissible"]],
}.items():
    for component in component_cols:
        component_rows.append({
            "admissibility": admissibility_name,
            "component": component,
            "tested_projections": len(subset),
            "positive_count": int((subset[component] > 0).sum()),
            "positive_rate": float((subset[component] > 0).mean()) if len(subset) else np.nan,
            "mean_magnitude": float(subset[component].mean()) if len(subset) else np.nan,
        })

component_rates = pd.DataFrame(component_rows)
component_rates

## 10. Domain-pair summary

In [ ]:
summary_rows = []

for (source_domain, target_domain), subset in df.groupby(
    ["source_domain", "target_domain"]
):
    for admissibility_name, local in {
        "weak": subset,
        "strong": subset[subset["strongly_admissible"]],
    }.items():
        summary_rows.append({
            "source_domain": source_domain,
            "target_domain": target_domain,
            "admissibility": admissibility_name,
            "tested_projections": len(local),
            "zero_information": int(local["zero_information_residue"].sum()),
            "zero_structural": int(local["zero_structural_residue"].sum()),
            "zero_reference": int(local["zero_reference_residue"].sum()),
            "zero_typed": int(local["zero_typed_residue"].sum()),
            "mean_information_residue": float(local["information_residue"].mean()) if len(local) else np.nan,
            "mean_structural_residue": float(local["structural_residue"].mean()) if len(local) else np.nan,
            "mean_typed_residue": float(local["typed_residue"].mean()) if len(local) else np.nan,
        })

projection_summary = pd.DataFrame(summary_rows)
projection_summary

## 11. Figures

In [ ]:
plot_data = test_results.copy()
plot_data["nonzero_residue_rate"] = 1.0 - plot_data["counterexample_rate"]

fig, ax = plt.subplots(figsize=(10, 6))
labels = (
    plot_data["admissibility"].astype(str)
    + " / "
    + plot_data["criterion"].astype(str)
)
ax.bar(labels, plot_data["nonzero_residue_rate"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Fraction with positive residue")
ax.set_title("Residue rate by criterion and admissibility class")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()

figure1_path = OUTPUT_DIR / "figure_residue_rates.png"
fig.savefig(figure1_path, dpi=180)
plt.show()

print(figure1_path)

In [ ]:
pair_plot = projection_summary[
    projection_summary["admissibility"] == "strong"
].copy()

pair_plot["pair"] = (
    pair_plot["source_domain"]
    + "→"
    + pair_plot["target_domain"]
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(pair_plot["pair"], pair_plot["zero_information"])
ax.set_ylabel("Zero-information-residue projections")
ax.set_title("Strong counterexamples by domain pair")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()

figure2_path = OUTPUT_DIR / "figure_zero_residue_by_domain_pair.png"
fig.savefig(figure2_path, dpi=180)
plt.show()

print(figure2_path)

## 12. Interpretation protocol

In [ ]:
def criterion_interpretation(criterion: str, verdict_text: str) -> str:
    if verdict_text == "FALSIFIED_IN_TESTED_MODEL_CLASS":
        return {
            "information": (
                "At least one admissible cross-domain map preserves all measured finite "
                "information and is invertible. Domain difference alone does not force "
                "information residue under this operational definition."
            ),
            "structural": (
                "At least one admissible cross-domain map preserves measured orientation "
                "and relation structure. Structural residue is not necessary in the tested class."
            ),
            "reference": (
                "At least one admissible cross-domain map requires no additional reference "
                "residue. Reference choice is not universally generated."
            ),
            "typed": (
                "A zero typed-residue projection exists despite domain difference. This "
                "would directly challenge the domain typing scheme itself."
            ),
        }[criterion]

    if verdict_text == "NOT_FALSIFIED_IN_TESTED_MODEL_CLASS":
        return {
            "information": (
                "Every tested admissible cross-domain map loses measured information or "
                "fails invertibility. The hypothesis survives for information residue in "
                "the bounded model class."
            ),
            "structural": (
                "Every tested admissible cross-domain map alters measured orientation or "
                "relation structure. The hypothesis survives for structural residue."
            ),
            "reference": (
                "Every tested admissible cross-domain map generates a reference mismatch "
                "or reference choice. The hypothesis survives for reference residue."
            ),
            "typed": (
                "Every tested cross-domain map has positive typed residue. Because domain "
                "difference contributes directly to this criterion, this is partly definitional "
                "and is not independent evidence."
            ),
        }[criterion]

    return "No admissible projections were available for this test."


test_results["interpretation"] = test_results.apply(
    lambda row: criterion_interpretation(row["criterion"], row["verdict"]),
    axis=1,
)

for _, row in test_results.iterrows():
    print(
        f"\n[{row['admissibility']} / {row['criterion']}]\n"
        f"Verdict: {row['verdict']}\n"
        f"{row['interpretation']}"
    )

## 13. Export all deliverables

In [ ]:
projection_records_path = OUTPUT_DIR / "projection_records.csv"
projection_summary_path = OUTPUT_DIR / "projection_summary.csv"
counterexamples_path = OUTPUT_DIR / "zero_residue_counterexamples.csv"
component_rates_path = OUTPUT_DIR / "residue_component_rates.csv"
test_results_path = OUTPUT_DIR / "hypothesis_test_results.csv"
block_manifest_path = OUTPUT_DIR / "block_manifest.csv"

df.to_csv(projection_records_path, index=False)
projection_summary.to_csv(projection_summary_path, index=False)
counterexamples.to_csv(counterexamples_path, index=False)
component_rates.to_csv(component_rates_path, index=False)
test_results.to_csv(test_results_path, index=False)
pd.DataFrame(block_manifest).to_csv(block_manifest_path, index=False)

print("CSV artifacts written.")

In [ ]:
primary_rows = test_results[
    test_results["admissibility"] == "strong"
].copy()

findings = {
    "notebook": 18,
    "title": "Every Domain Projection Necessarily Generates Residue",
    "hypothesis": (
        "Every genuine projection between distinct domains necessarily "
        "generates residue."
    ),
    "scope": {
        "claim_scope": "bounded finite relational model class",
        "source_domains": CONFIG["source_domains"],
        "target_domains": CONFIG["target_domains"],
        "source_sizes": CONFIG["source_sizes"],
        "target_sizes": CONFIG["target_sizes"],
        "projection_classes": ["weak", "strong"],
        "residue_criteria": list(CRITERIA.keys()),
    },
    "primary_results_strong_admissibility": primary_rows[
        [
            "criterion",
            "tested_projections",
            "zero_residue_counterexamples",
            "counterexample_rate",
            "verdict",
            "interpretation",
        ]
    ].to_dict(orient="records"),
    "validation": VALIDATION,
    "important_methodological_result": (
        "The truth value of the universal hypothesis depends on the operational "
        "definition of residue. Typed-domain residue includes domain mismatch by "
        "definition and therefore cannot independently establish the hypothesis."
    ),
    "falsification_rule": (
        "One strongly admissible zero-residue cross-domain projection falsifies "
        "the universal claim for the tested residue criterion and model class."
    ),
    "limitations": [
        "Finite organizations of bounded size were tested.",
        "Target organization blocks above the safety threshold used deterministic stratification.",
        "The residue vector is operational and may not exhaust RT residue.",
        "A surviving universal claim is not proven outside the tested model class.",
        "Typed residue is partly definitional because domain mismatch contributes directly.",
    ],
    "recommended_next_step": {
        "notebook": 19,
        "title": "Necessary and Sufficient Conditions for Projection Residue",
        "question": (
            "Which properties of a projection force positive information or "
            "structural residue, independent of domain labels?"
        ),
    },
}

findings_path = OUTPUT_DIR / "findings18.json"
findings_path.write_text(
    json.dumps(findings, indent=2),
    encoding="utf-8",
)

run_manifest = {
    "notebook": 18,
    "seed": SEED,
    "python": sys.version,
    "platform": platform.platform(),
    "elapsed_seconds": elapsed,
    "projection_record_count": len(df),
    "strong_projection_count": int(df["strongly_admissible"].sum()),
    "configuration": CONFIG,
    "target_safety_threshold": MAX_TARGET_ORGS_PER_BLOCK,
    "blocks": block_manifest,
    "artifacts": [
        str(projection_records_path),
        str(projection_summary_path),
        str(counterexamples_path),
        str(component_rates_path),
        str(test_results_path),
        str(block_manifest_path),
        str(figure1_path),
        str(figure2_path),
        str(findings_path),
    ],
}

manifest_path = OUTPUT_DIR / "run_manifest18.json"
manifest_path.write_text(
    json.dumps(run_manifest, indent=2),
    encoding="utf-8",
)

print(json.dumps(findings, indent=2))

## 14. Package outputs

In [ ]:
zip_path = OUTPUT_DIR / "RT_Notebook_18_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path == zip_path:
            continue
        if path.is_file():
            zf.write(path, arcname=path.name)

print("Output archive:", zip_path.resolve())
print("Archive size:", zip_path.stat().st_size, "bytes")

# Final Decision Standard

The notebook’s final result must be read criterion by criterion.

A defensible conclusion has the form:

> The universal residue hypothesis was **falsified** or **not falsified** within the declared finite model class under a specified operational residue criterion.

It must **not** be reported merely as:

> Every projection generates residue.

unless no counterexample exists and a separate theorem establishes necessity beyond the finite campaign.

# Notebook 19 Trigger

Proceed to Notebook 19 after examining the strongest non-definitional criterion:

- information residue, and
- structural residue.

Notebook 19 should derive necessary and sufficient conditions for positive residue rather than repeat the universal search.